### วิเคราะห์และเตรียมข้อมูล

In [ ]:
import pandas as pd

df = pd.read_csv("patient_symptoms.csv")
df.head()


,gender,age,summary,search_term
0,male,28,"{""diseases"": [], ""procedures"": [], ""no_symptom...","มีเสมหะ, ไอ"
1,male,27,"{""diseases"": [], ""procedures"": [], ""no_symptom...","ไอ, น้ำมูกไหล"
2,female,26,"{""diseases"": [], ""procedures"": [], ""no_symptom...",ปวดท้อง
3,male,42,"{""diseases"": [], ""procedures"": [], ""no_symptom...",น้ำมูกไหล
4,female,40,"{""diseases"": [], ""procedures"": [], ""no_symptom...",ตาแห้ง


In [ ]:
df["gender"] = df["gender"].map({"male": 0, "female": 1})

# ฟังก์ชันแปลงป็น list
def clean_symptoms(symptom_str):
    if pd.isnull(symptom_str):
        return []
    return list(set([s.strip() for s in symptom_str.split(",") if s.strip()]))

df["symptom_list"] = df["search_term"].apply(clean_symptoms)

df[["gender", "age", "symptom_list"]].head()


,gender,age,symptom_list
0,0,28,"[มีเสมหะ, ไอ]"
1,0,27,"[น้ำมูกไหล, ไอ]"
2,1,26,[ปวดท้อง]
3,0,42,[น้ำมูกไหล]
4,1,40,[ตาแห้ง]


In [ ]:
from collections import Counter

all_symptoms = sum(df["symptom_list"].tolist(), [])

symptom_freq = Counter(all_symptoms)
symptom_freq.most_common(10)


[('ไอ', 173),
 ('มีเสมหะ', 104),
 ('เจ็บคอ', 102),
 ('น้ำมูกไหล', 86),
 ('ไข้', 78),
 ('ปวดท้อง', 49),
 ('ผื่น', 45),
 ('ท้องเสีย', 32),
 ('คัน', 31),
 ('คัดจมูก', 23)]

In [ ]:
df_cleaned = df[["gender", "age", "symptom_list"]].copy()

df_cleaned.to_csv("preprocessed_data.csv", index=False)


### สร้างฟังก์ชันแนะนำอาการ

In [5]:
import pandas as pd

df = pd.read_csv("preprocessed_data.csv")
df.head()


,gender,age,symptom_list
0,0,28,"['มีเสมหะ', 'ไอ']"
1,0,27,"['น้ำมูกไหล', 'ไอ']"
2,1,26,['ปวดท้อง']
3,0,42,['น้ำมูกไหล']
4,1,40,['ตาแห้ง']


In [ ]:
from collections import Counter

def recommend_symptoms(gender, age, user_symptoms, df, top_n=5):
    # แปลงอาการให้เป็น set
    user_symptoms_set = set(user_symptoms)
    
    matched_symptoms = []

    for _, row in df.iterrows():
        if not isinstance(row["symptom_list"], str):
            continue

        patient_symptoms = eval(row["symptom_list"])
        patient_symptoms_set = set(patient_symptoms)

        if user_symptoms_set & patient_symptoms_set:
            matched_symptoms.extend(patient_symptoms)

    new_symptoms = list(set(matched_symptoms) - user_symptoms_set)

    sorted_symptoms = [s for s, _ in Counter(new_symptoms).most_common(top_n)]
    
    return sorted_symptoms


In [ ]:
# input ทดสอบ
user_input = {
    "gender": 1,
    "age": 25,
    "symptoms": ["ไอ", "มีเสมหะ"]
}

recommendations = recommend_symptoms(
    gender=user_input["gender"],
    age=user_input["age"],
    user_symptoms=user_input["symptoms"],
    df=df,
    top_n=5
)

print("แนะนำอาการเพิ่มเติม:", recommendations)


แนะนำอาการเพิ่มเติม: ['คอแห้งคอแดง', 'น้ำมูกไหลคัดจมูก', 'กลืนเจ็บ', 'มีเสมหะน้ำมูกไหล', 'จามบ่อย']


### แนะนำอาการด้วย KMeans + Multi-Label Encoding

In [ ]:
import pandas as pd

df = pd.read_csv("preprocessed_data.csv")

# แปลง str เป็น list
df["symptom_list"] = df["symptom_list"].apply(eval)


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
X_symptom = mlb.fit_transform(df["symptom_list"])

X = pd.concat([
    df[["gender", "age"]].reset_index(drop=True),
    pd.DataFrame(X_symptom, columns=mlb.classes_)
], axis=1)


In [ ]:
from sklearn.cluster import KMeans

k = 5  # จัดกลุ่ม 5 กลุ่ม
kmeans = KMeans(n_clusters=k, random_state=42)
df["cluster"] = kmeans.fit_predict(X)


In [ ]:
from collections import Counter

def recommend_from_cluster(gender, age, user_symptoms, top_n=5):
    # input vector
    input_vec = [0] * len(mlb.classes_)
    for i, s in enumerate(mlb.classes_):
        if s in user_symptoms:
            input_vec[i] = 1

    input_data = [[gender, age] + input_vec]

    # หา cluster
    cluster_id = kmeans.predict(input_data)[0]

    # กรองกลุ่มเดียวกัน
    same_group = df[df["cluster"] == cluster_id]

    # รวมอาการทั้งหมดในกลุ่มนั้น
    all_symptoms = sum(same_group["symptom_list"].tolist(), [])

    # ลบอาการที่ผู้ใช้มีแล้ว
    suggestions = list(set(all_symptoms) - set(user_symptoms))

    # จัดเรียงตามความถี่
    return [s for s, _ in Counter(suggestions).most_common(top_n)]


In [ ]:
# input ทดสอบ
user_input = {
    "gender": 1,  # female
    "age": 30,
    "symptoms": ["ไอ", "มีเสมหะ"]
}

recommendations = recommend_from_cluster(
    gender=user_input["gender"],
    age=user_input["age"],
    user_symptoms=user_input["symptoms"],
    top_n=5
)

print("แนะนำอาการเพิ่มเติม:", recommendations)


แนะนำอาการเพิ่มเติม: ['ปวดข้อเท้า', 'Weight lossstomachache', 'น้ำมูกไหลคัดจมูก', 'มึนศีรษะ', 'armpit lump']


d:\agnos\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
